# AIC 2026 - PaddleOCR A100 Batch 32 from Azure

This is the Azure version of the isolated A100 Batch 32 notebook. It keeps PaddleOCR in a separate venv/process and reads exact Jina frame identity from `global_ids.parquet`.

```text
Azure keyframes + Azure global_ids.parquet
        -> download one video to Colab SSD
        -> PaddleOCR GPU batch 32
        -> per-video checkpoint on Azure
        -> delete local frames
```

No Google Drive, ZIP extraction, legacy FAISS counter, or `map-keyframes.zip` is used as an input. Google Drive is mounted only to back up checkpoints and OCR snapshots. A completed OCR record already has Jina `vector_id`, `frame_path`, `timestamp`, and `source_frame_idx`.

## 1. Verify the A100 runtime

In [ ]:
!nvidia-smi

## 2. Create the isolated PaddleOCR environment

In [ ]:
import subprocess
import sys
from pathlib import Path

ENV_DIR = Path('/content/paddle_ocr_env')
PYTHON = ENV_DIR / 'bin/python'

if not PYTHON.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    subprocess.check_call([sys.executable, '-m', 'uv', 'venv', str(ENV_DIR)])
    subprocess.check_call([
        sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(PYTHON),
        'paddlepaddle-gpu==3.3.1', '--index-url',
        'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
    ])
    subprocess.check_call([
        sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(PYTHON),
        'paddleocr==3.7.0', 'azure-storage-blob', 'pandas', 'pyarrow', 'tqdm'
    ])

print('Isolated worker Python:', PYTHON)

## 3. Configure Azure and load the worker

In [ ]:
import os
from pathlib import Path
from google.colab import drive

try:
    from google.colab import userdata
    AZURE_STORAGE_CONNECTION_STRING = userdata.get('AZURE_STORAGE_CONNECTION_STRING')
except Exception:
    AZURE_STORAGE_CONNECTION_STRING = os.environ.get('AZURE_STORAGE_CONNECTION_STRING', '')

if not AZURE_STORAGE_CONNECTION_STRING:
    raise RuntimeError('Create the Colab secret AZURE_STORAGE_CONNECTION_STRING, then rerun this cell.')

KEYFRAMES_CONTAINER = 'keyframes'
EMBEDDINGS_CONTAINER = 'embeddings'
RESULTS_CONTAINER = 'metadata'
GLOBAL_IDS_BLOB = 'indexes/fine_keyframes_jina_clip_v2_1024d_v2/jina/global_ids.parquet'
RUN_NAME = 'fine_keyframes_jina_paddleocr_smoke_v1'
ONLY_NAMESPACES = ''  # Example: 'L21_a,L22_a'. Empty means all namespaces.
MAX_VIDEOS = 1        # Safe default: exactly one video for the smoke test.
# Full run only: set RUN_NAME = 'fine_keyframes_jina_paddleocr_v1' and MAX_VIDEOS = None.

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/AIC_2026/OCR_Azure_B32')
DRIVE_OUTPUT_DIR = DRIVE_ROOT / RUN_NAME
DRIVE_SNAPSHOT_EVERY_VIDEOS = 10  # Also writes each individual video checkpoint immediately.
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WORKER = Path('/content/ocr_azure_worker.py')
if not WORKER.exists():
    from google.colab import files
    print('Upload scripts/data_extraction/new/ocr_azure_worker.py from this repository.')
    uploaded = files.upload()
    if 'ocr_azure_worker.py' not in uploaded:
        raise RuntimeError('Expected ocr_azure_worker.py in the uploaded files.')
    WORKER.write_bytes(uploaded['ocr_azure_worker.py'])

print('Worker:', WORKER)
print('Azure run prefix:', f'ocr-runs/{RUN_NAME}')
print('Google Drive backup:', DRIVE_OUTPUT_DIR)

## 4. Smoke test the isolated GPU environment

In [ ]:
import subprocess
test_code = """
import paddle
print('Paddle:', paddle.__version__)
print('CUDA compiled:', paddle.is_compiled_with_cuda())
print('GPU count:', paddle.device.cuda.device_count())
"""
subprocess.run([str(PYTHON), '-c', test_code], check=True)

## 5. Run Azure OCR with batch size 32

In [ ]:
import subprocess
import os

env = os.environ.copy()
env['AZURE_STORAGE_CONNECTION_STRING'] = AZURE_STORAGE_CONNECTION_STRING
env['PADDLE_PDX_MODEL_SOURCE'] = 'BOS'
env['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'
env['PYTHONUNBUFFERED'] = '1'

cmd = [
    str(PYTHON), '-u', str(WORKER),
    '--keyframes-container', KEYFRAMES_CONTAINER,
    '--embeddings-container', EMBEDDINGS_CONTAINER,
    '--results-container', RESULTS_CONTAINER,
    '--global-ids-blob', GLOBAL_IDS_BLOB,
    '--run-name', RUN_NAME,
    '--batch-size', '32',
    '--download-workers', '16',
    '--log-every-videos', '1',
    '--drive-output-dir', str(DRIVE_OUTPUT_DIR),
    '--drive-snapshot-every-videos', str(DRIVE_SNAPSHOT_EVERY_VIDEOS),
]
if ONLY_NAMESPACES:
    cmd += ['--only-namespaces', ONLY_NAMESPACES]
if MAX_VIDEOS is not None:
    cmd += ['--max-videos', str(MAX_VIDEOS)]

from IPython.display import clear_output
print('Running OCR worker. The display refreshes after each video; the full log is saved separately.')
LOG_PATH = DRIVE_OUTPUT_DIR / 'ocr_azure_worker.log'
process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with LOG_PATH.open('w', encoding='utf-8') as log_file:
    for line in process.stdout:
        log_file.write(line)
        if 'Progress:' in line:
            clear_output(wait=True)
            print('PaddleOCR Azure run')
            print(line.strip())
            print('Full log:', LOG_PATH)
        elif 'ERROR' in line or 'Traceback' in line:
            print(line, end='', flush=True)
if process.wait() != 0:
    raise RuntimeError(f'OCR worker failed. Read {LOG_PATH}; completed videos remain checkpointed on Azure.')
print('Full worker log:', LOG_PATH)

## 6. Merge completed video checkpoints into one Jina OCR JSON

Run this only after the full corpus is complete. It refuses smoke tests or split-only runs. Set both `ONLY_NAMESPACES = ''` and `MAX_VIDEOS = None` in Cell 3 first.

In [ ]:
import subprocess
import os

env = os.environ.copy()
env['AZURE_STORAGE_CONNECTION_STRING'] = AZURE_STORAGE_CONNECTION_STRING
cmd = [
    str(PYTHON), '-u', str(WORKER), '--merge-only',
    '--keyframes-container', KEYFRAMES_CONTAINER,
    '--embeddings-container', EMBEDDINGS_CONTAINER,
    '--results-container', RESULTS_CONTAINER,
    '--global-ids-blob', GLOBAL_IDS_BLOB, '--run-name', RUN_NAME,
    '--drive-output-dir', str(DRIVE_OUTPUT_DIR),
]
if ONLY_NAMESPACES:
    cmd += ['--only-namespaces', ONLY_NAMESPACES]
if MAX_VIDEOS is not None:
    cmd += ['--max-videos', str(MAX_VIDEOS)]
subprocess.run(cmd, env=env, check=True)
print(f'Final output: {RESULTS_CONTAINER}/ocr-runs/{RUN_NAME}/ocr_results_jina.json')